In [1]:
import tiktoken
import json
import re

# tiktoken library

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

sentence = "This is \n some text"

tiktoken_tokens = tokenizer.encode(sentence)
text = tokenizer.decode(tiktoken_tokens)

print("Sentence:", sentence)
print("Tokens:", tiktoken_tokens)
print("Decoded:", text)

assert sentence == text, "Decoded text does not match the original sentence."

Sentence: This is 
 some text
Tokens: [1212, 318, 220, 198, 617, 2420]
Decoded: This is 
 some text


# my tokenizer

## Encode

In [3]:
with open("encoder.json", "r", encoding="utf-8") as f:
    vocab = json.load(f)
    id_to_token = {int(v): k for k, v in vocab.items()}
    token_to_id = {k: int(v) for k, v in vocab.items()}

    token_to_id["\n"] = 198
    token_to_id["\r"] = 201

In [ ]:
def text_to_tokens(text:str) -> list[str]:
    """
    Convert given text into a list of tokens
    """
    tokens = []
    parts = re.split(r'(\r\n|\r|\n)', text)
    for part in parts:
        if part == "":
            continue
        if part == "\r\n":
            tokens.append("\r")
            tokens.append("\n")
            continue
        if part == "\r":
            tokens.append("\r")
            continue
        if part == "\n":
            tokens.append("\n")
            continue

        s = re.split(r'(\S+)', part)
        
        if s[0] == "":
            s = s[1:]
        if s[-1] == "":
            s = s[:-1]

        for i, string in enumerate(s):

            if string[0] == " ":
                k = len(string)-1 if i < len(s)-1 else len(string)
                tokens += ["Ġ"] * k

            elif i == 0:
                tokens.append(string)
                
            else:
                tokens.append("Ġ"+string)
        
    return tokens

def bpe_tokenize(token:str) -> list[int]:
    """
    Tokenize a single token using BPE
    """
    raise NotImplementedError("BPE tokenization is not implemented yet.")

def encode(text:str) -> list[int]:
    """
    Encode given text into a list of token IDs
    """
    token_ids = []
    tokens = []

    tokens = text_to_tokens(text)

    for token in tokens:
        if token not in token_to_id:
            token_ids += bpe_tokenize(token)
        else:
            token_ids.append(token_to_id[token])
            

    return token_ids

In [5]:
encoded_tokens = encode("This is \n some text")
encoded_tokens

[1212, 318, 220, 198, 617, 2420]

## Decode

In [6]:
def decode(token_ids: list[int]) -> str:
    """
    Decode given list of token IDs into text
    """
    tokens = []
    for token_id in token_ids:
        if token_id not in id_to_token:
            raise ValueError(f"Token ID '{token_id}' not found in vocabulary.")

        token = id_to_token[token_id]

        if token_id == 198:
            tokens.append("\n")
        elif token_id == 201:
            tokens.append("\r")
            
        elif token[0] == "Ġ":
            tokens.append(" " + token[1:])
        else:
            tokens.append(token)
        
    return "".join(tokens)

In [7]:
decoded_text = decode(encoded_tokens)
decoded_text

'This is \n some text'

In [8]:
sentence = "This is \n some text"

my_tokens = encode(sentence)
text = decode(my_tokens)

print("Sentence:", sentence)
print("Tokens:", my_tokens)
print("Decoded:", text)

assert sentence == text, "Decoded text does not match the original sentence."

assert my_tokens == tiktoken_tokens, "My tokens do not match the TikToken tokens."

Sentence: This is 
 some text
Tokens: [1212, 318, 220, 198, 617, 2420]
Decoded: This is 
 some text
